# PyTorch classification of single-cell SMAD time courses

This notebook classifies control, high TGF-beta and high GDF11 trajectories. The model combines a raw-signal branch with a derivative branch so that absolute level and temporal transitions can contribute independently.

## 1. Setup

The repository must be installed once with `python -m pip install -e ".[torch]"`. The model and training loop live in the project package; this notebook only assembles the experiment.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import train_test_split

from deeplearning_examples.timecourse import load_smad_classification_data
from pytorch_timecourse_classification.model import TimecourseClassifier
from pytorch_timecourse_classification.training import TrainingConfig, train

## 2. Data and target representation

The shared loader returns zero-based class labels and exposes both channel conventions. PyTorch expects `(samples, channels, timepoints)`.

In [ ]:
dataset = load_smad_classification_data()
features = torch.from_numpy(dataset.channels_first).float()
targets = torch.from_numpy(dataset.targets).long()

print(f"features: {tuple(features.shape)}")
print(f"targets:  {tuple(targets.shape)}")
print(dict(zip(dataset.class_names, np.bincount(dataset.targets))))

### Condition-level trajectory summaries

The median and interquartile range show the population response without suggesting that every cell follows the same trajectory.

In [ ]:
figure, axes = plt.subplots(1, len(dataset.class_names), figsize=(11, 3), sharey=True)
for class_index, (class_name, axis) in enumerate(zip(dataset.class_names, axes)):
    values = dataset.trajectories[dataset.targets == class_index]
    lower, median, upper = np.percentile(values, [25, 50, 75], axis=0)
    axis.fill_between(dataset.times, lower, upper, alpha=0.25)
    axis.plot(dataset.times, median, linewidth=2)
    axis.set_title(class_name)
    axis.set_xlabel("time")
axes[0].set_ylabel("SMAD signal")
figure.tight_layout()

## 3. Held-out test split

The training function creates its own stratified validation subset. This outer split remains untouched until final evaluation.

In [ ]:
indices = np.arange(len(features))
train_indices, test_indices = train_test_split(
    indices, test_size=0.20, random_state=42, stratify=dataset.targets
)
x_train, x_test = features[train_indices], features[test_indices]
y_train, y_test = targets[train_indices], targets[test_indices]

## 4. Model and optimization

`TimecourseClassifier` returns logits. `train` applies cross-entropy, learning-rate decay, early stopping and restoration of the best validation weights.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TimecourseClassifier(
    input_length=features.shape[-1], num_classes=len(dataset.class_names)
)
history = train(
    model,
    x_train,
    y_train,
    config=TrainingConfig(epochs=100, batch_size=512, patience=10),
    device=device,
)

## 5. Training diagnostics

A growing gap between training and validation curves indicates overfitting; the reported best epoch is the checkpoint restored by `train`.

In [ ]:
epochs = np.arange(1, len(history.training_loss) + 1)
figure, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].plot(epochs, history.training_loss, label="train")
axes[0].plot(epochs, history.validation_loss, label="validation")
axes[0].axvline(history.best_epoch, color="black", linestyle=":", label="best epoch")
axes[0].set(title="Loss", xlabel="epoch")
axes[1].plot(epochs, history.training_accuracy, label="train")
axes[1].plot(epochs, history.validation_accuracy, label="validation")
axes[1].set(title="Accuracy", xlabel="epoch")
for axis in axes: axis.legend()
figure.tight_layout()

## 6. Evaluation on held-out cells

Accuracy is complemented by class-wise precision and recall because similar overall scores can hide a weak biological condition.

In [ ]:
model.eval()
with torch.no_grad():
    predictions = model(x_test.to(device)).argmax(1).cpu().numpy()
print(classification_report(y_test.numpy(), predictions, target_names=dataset.class_names))
ConfusionMatrixDisplay.from_predictions(
    y_test.numpy(), predictions, display_labels=dataset.class_names, xticks_rotation=25
)
plt.tight_layout()

## 7. Save the trained state

Saving `state_dict` keeps the checkpoint independent of notebook state.

In [ ]:
checkpoint_path = "timecourse_classifier.pth"
torch.save(model.state_dict(), checkpoint_path)
print(checkpoint_path)